# Diffusion Models per la generazione di scenari di rischio finanziario
### Confronto con GARCH(1,1)-t e QuantGAN su dati reali (S&P 500)

**Tesi magistrale — Statistica e Data Science**

Questo notebook implementa e confronta tre modelli generativi per i rendimenti giornalieri dell'indice S&P 500:

| Modello | Tipo | Ruolo |
|---|---|---|
| **GARCH(1,1)-t** | econometrico parametrico | benchmark classico |
| **QuantGAN** | deep, avversariale (TCN) | benchmark deep |
| **Diffusion (DDPM)** | deep, score-based | modello proposto |

Il confronto avviene su due piani:
1. **Realismo** — i *fatti stilizzati* (code grasse, volatility clustering, assenza di autocorrelazione nei rendimenti);
2. **Utilità per il risk management** — stima di **VaR** e **CVaR** a 1 giorno e *backtesting* out-of-sample (Kupiec + Christoffersen).

> **Come eseguirlo su Colab.** `Runtime ▸ Change runtime type ▸ GPU` (consigliato per QuantGAN e Diffusion). Poi `Runtime ▸ Run all`. La prima cella installa `yfinance` e `arch`; `torch` è già presente su Colab.


## 0. Setup e riproducibilità

In [ ]:
# Installazione pacchetti non presenti di default su Colab
!pip -q install yfinance arch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", DEVICE)

# ---- Iperparametri globali (modificabili) ----
TICKER      = "^GSPC"       # S&P 500. Per mercati energetici: "CL=F" (petrolio), "NG=F" (gas)
START       = "1990-01-01"
SPLIT_DATE  = "2018-12-31"  # train = ...->2018 | test = 2019->oggi
L           = 64            # lunghezza finestra (giorni) per i modelli generativi
ALPHA       = 0.01          # livello VaR/CVaR = 99%
LAMBDA_EWMA = 0.94          # RiskMetrics


## 1. Dati reali (yfinance) e rendimenti logaritmici

Scarichiamo i prezzi *adjusted close* e li trasformiamo in **rendimenti logaritmici**
$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right),$$
che rendono la serie (quasi) stazionaria — condizione necessaria per tutti i modelli.


In [ ]:
import yfinance as yf

raw = yf.download(TICKER, start=START, auto_adjust=True, progress=False)
close = raw["Close"]
if isinstance(close, pd.DataFrame):      # yfinance recente restituisce colonne MultiIndex
    close = close.iloc[:, 0]
price = close.dropna()

ret = np.log(price / price.shift(1)).dropna()
ret.name = "logret"

print(f"Osservazioni: {len(ret)}  |  dal {ret.index.min().date()} al {ret.index.max().date()}")
ret.tail()


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(price.index, price.values, lw=0.7); ax[0].set_title(f"{TICKER} — prezzo (adjusted close)")
ax[1].plot(ret.index, ret.values, lw=0.5, color="crimson"); ax[1].set_title("Rendimenti logaritmici giornalieri")
plt.tight_layout(); plt.show()


## 2. Fatti stilizzati sui dati reali

Definiamo una funzione che misura le proprietà che ogni buon generatore dovrà riprodurre:

- **curtosi > 3** → code più grasse della normale;
- **ACF dei rendimenti ≈ 0** → i rendimenti non sono prevedibili linearmente;
- **ACF dei rendimenti² decrescente lenta** → *volatility clustering*;
- **asimmetria negativa** → l'effetto leva (i crolli sono più bruschi dei rialzi).


In [ ]:
from statsmodels.tsa.stattools import acf

def stylized_facts(x, name="serie", nlags=10):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    out = {
        "media":      x.mean(),
        "dev.std":    x.std(),
        "asimmetria": stats.skew(x),
        "curtosi":    stats.kurtosis(x, fisher=False),   # 3 = normale
        "ACF r (lag1)":   acf(x, nlags=nlags, fft=True)[1],
        "ACF r^2 (lag1)": acf(x**2, nlags=nlags, fft=True)[1],
        "ACF r^2 (lag5)": acf(x**2, nlags=nlags, fft=True)[5],
    }
    return pd.Series(out, name=name)

sf_real = stylized_facts(ret.values, "S&P500 (reale)")
sf_real.to_frame().T


In [ ]:
# --- Cella di TEST: sui dati reali le proprietà attese devono valere ---
assert sf_real["curtosi"] > 3,            "Atteso: code grasse (curtosi > 3)"
assert abs(sf_real["ACF r (lag1)"]) < 0.1, "Atteso: rendimenti quasi non autocorrelati"
assert sf_real["ACF r^2 (lag1)"] > sf_real["ACF r (lag1)"], "Atteso: volatility clustering"
print("OK — i dati reali mostrano i fatti stilizzati attesi.")


## 3. Split temporale, standardizzazione e finestre scorrevoli

**Regola anti-leakage:** lo split è **temporale** (mai casuale) e media/deviazione standard si stimano **solo sul train**.

Da un'unica storia otteniamo migliaia di sotto-serie con finestre scorrevoli di lunghezza `L`, il materiale su cui i modelli generativi imparano la distribuzione delle traiettorie.


In [ ]:
r_train = ret.loc[:SPLIT_DATE]
r_test  = ret.loc[SPLIT_DATE:].iloc[1:]        # evita la sovrapposizione sul giorno di split

mu, sd = r_train.mean(), r_train.std()         # statistiche SOLO dal train
z_train = (r_train - mu) / sd                  # rendimenti standardizzati

def make_windows(z, L):
    z = np.asarray(z, dtype=np.float32)
    N = len(z) - L + 1
    return np.stack([z[i:i+L] for i in range(N)])

W = make_windows(z_train.values, L)            # (N, L)
print(f"train: {len(r_train)}  |  test: {len(r_test)}")
print(f"finestre di training: {W.shape}  (N campioni x L)")
print(f"mu_train = {mu:.6f}   sd_train = {sd:.6f}")


In [ ]:
# --- Cella di TEST: forme e standardizzazione ---
assert W.shape[1] == L
assert W.shape[0] == len(z_train) - L + 1
assert abs(z_train.mean()) < 1e-6 and abs(z_train.std() - 1) < 1e-2
assert r_test.index.min() > r_train.index.max(), "Il test deve venire DOPO il train"
print("OK — split temporale corretto e finestre ben formate.")


## 4. Protocollo unificato di VaR/CVaR e backtesting

Per confrontare i tre modelli in modo equo definiamo un protocollo comune.
Ogni modello fornisce due ingredienti:

1. una **forma** della distribuzione delle innovazioni standardizzate $z$ (da cui i quantili $q_\alpha$ e l'expected shortfall $es_\alpha$ *empirici*);
2. una **scala condizionale** $\sigma_t$.

Il VaR e il CVaR condizionali a 1 giorno sono
$$\text{VaR}_t = \mu + \sigma_t\, q_\alpha(z), \qquad \text{CVaR}_t = \mu + \sigma_t\, es_\alpha(z).$$

- Per **GARCH-t** la scala $\sigma_t$ è la sua stessa volatilità condizionale e $z\sim t_\nu$ standardizzata.
- Per **QuantGAN** e **Diffusion** la forma $z$ è ricavata dai campioni generati, mentre la scala è la volatilità **EWMA (RiskMetrics)**. In questo modo i generatori contribuiscono la *forma delle code*, l'EWMA la *scala condizionale* → i tre VaR sono tutti condizionali e confrontabili.

**Backtesting.** Una *violazione* è $r_t < \text{VaR}_t$. Verifichiamo:
- **Kupiec (POF)** — la frequenza di violazioni è coerente con $\alpha$?
- **Christoffersen (indipendenza)** — le violazioni sono indipendenti (non a grappoli)?
- **Copertura condizionale (CC)** — i due test combinati.


In [ ]:
from scipy.stats import chi2

def ewma_sigma(r, lam=LAMBDA_EWMA):
    r = np.asarray(r, dtype=float)
    var = np.empty(len(r)); var[0] = r.var()
    for t in range(1, len(r)):
        var[t] = lam * var[t-1] + (1 - lam) * r[t-1]**2
    return np.sqrt(var)

def tail_shape(z, alpha=ALPHA):
    # quantile ed expected shortfall EMPIRICI di innovazioni standardizzate a varianza ~1
    z = np.asarray(z, dtype=float); z = z[np.isfinite(z)]
    z = (z - z.mean()) / z.std()          # forza varianza unitaria: la scala la dà sigma_t
    q  = np.quantile(z, alpha)
    es = z[z <= q].mean()
    return q, es

def var_cvar_series(sigma_t, q, es, mu=0.0):
    return mu + sigma_t * q, mu + sigma_t * es

def kupiec(viol, alpha=ALPHA):
    viol = np.asarray(viol, dtype=int); n = len(viol); x = int(viol.sum()); pih = x / n
    def ll(p, a, b):
        p = min(max(p, 1e-12), 1 - 1e-12)
        return a * np.log(1 - p) + b * np.log(p)
    LR = -2 * (ll(alpha, n - x, x) - ll(pih, n - x, x))
    return dict(viol=x, n=n, freq=pih, LR=LR, pvalue=1 - chi2.cdf(LR, 1))

def christoffersen_ind(viol):
    v = np.asarray(viol, dtype=int)
    n00 = n01 = n10 = n11 = 0
    for i in range(1, len(v)):
        a, b = v[i-1], v[i]
        if   a == 0 and b == 0: n00 += 1
        elif a == 0 and b == 1: n01 += 1
        elif a == 1 and b == 0: n10 += 1
        else:                   n11 += 1
    def ll(p, a, b):
        if p <= 0 or p >= 1: return 0.0
        return a * np.log(1 - p) + b * np.log(p)
    pi01 = n01 / (n00 + n01) if (n00 + n01) else 0.0
    pi11 = n11 / (n10 + n11) if (n10 + n11) else 0.0
    pi   = (n01 + n11) / max(n00 + n01 + n10 + n11, 1)
    LR = -2 * (ll(pi, n00 + n10, n01 + n11) - (ll(pi01, n00, n01) + ll(pi11, n10, n11)))
    return dict(LR=LR, pvalue=1 - chi2.cdf(LR, 1))

def backtest(r_test, VaR, CVaR, alpha=ALPHA, name="modello"):
    r_test = np.asarray(r_test, dtype=float)
    viol = (r_test < VaR).astype(int)
    kp = kupiec(viol, alpha); ci = christoffersen_ind(viol)
    LR_cc = kp["LR"] + ci["LR"]; p_cc = 1 - chi2.cdf(LR_cc, 2)
    breaches = viol.astype(bool)
    cvar_err = np.nan
    if breaches.sum() > 0:                       # errore medio del CVaR sulle code osservate
        cvar_err = np.mean(r_test[breaches] - CVaR[breaches])
    return pd.Series({
        "violazioni": kp["viol"], "attese": round(alpha * kp["n"], 1),
        "freq %": 100 * kp["freq"], "Kupiec p": kp["pvalue"],
        "Christoff. p": ci["pvalue"], "CC p": p_cc,
        "CVaR mean-err": cvar_err,
    }, name=name)

sigma_ewma_full = ewma_sigma(ret.values)                       # su tutta la serie
sigma_ewma_test = pd.Series(sigma_ewma_full, index=ret.index).loc[r_test.index].values
print("EWMA sigma sul test:", sigma_ewma_test.shape, "min/max:",
      round(sigma_ewma_test.min(), 4), round(sigma_ewma_test.max(), 4))


## 5. Benchmark 1 — GARCH(1,1)-t

$$\sigma_t^2 = \omega + \alpha\,\varepsilon_{t-1}^2 + \beta\,\sigma_{t-1}^2,\qquad
\varepsilon_t = \sigma_t z_t,\; z_t \sim t_\nu.$$

Stimiamo i parametri **sul solo train**, poi filtriamo la volatilità condizionale in avanti sul test
usando la ricorsione con i parametri stimati (nessun refit sul test → nessun leakage).


In [ ]:
from arch import arch_model

am  = arch_model(r_train, mean="Constant", vol="GARCH", p=1, q=1, dist="t", rescale=False)
res = am.fit(disp="off")
print(res.summary().tables[1])

w   = res.params["omega"]; a1 = res.params["alpha[1]"]; b1 = res.params["beta[1]"]
nu  = res.params["nu"];    mu_g = res.params["mu"]
print(f"\npersistenza alpha+beta = {a1 + b1:.4f}   |   gradi di liberta' nu = {nu:.2f}")


In [ ]:
# Ricorsione della varianza condizionale sull'INTERA serie, con i parametri stimati sul train
r_all = ret.values.astype(float)
sig2  = np.empty(len(r_all)); sig2[0] = w / max(1 - a1 - b1, 1e-6)   # varianza incondizionata
for t in range(1, len(r_all)):
    eps = r_all[t-1] - mu_g
    sig2[t] = w + a1 * eps**2 + b1 * sig2[t-1]
sigma_garch = pd.Series(np.sqrt(sig2), index=ret.index)
sigma_garch_test = sigma_garch.loc[r_test.index].values

# forma delle innovazioni: t di Student standardizzata a varianza unitaria
scale_t = np.sqrt((nu - 2) / nu)
z_garch = stats.t.rvs(nu, size=200_000, random_state=SEED) * scale_t
q_g, es_g = tail_shape(z_garch)
VaR_g, CVaR_g = var_cvar_series(sigma_garch_test, q_g, es_g, mu=mu_g)

# path simulato per i fatti stilizzati (lunghezza pari al train)
def simulate_garch(n, seed=SEED):
    rng = np.random.default_rng(seed)
    s2 = w / max(1 - a1 - b1, 1e-6); out = np.empty(n)
    for t in range(n):
        z = rng.standard_t(nu) * scale_t
        e = np.sqrt(s2) * z; out[t] = mu_g + e
        s2 = w + a1 * e**2 + b1 * s2
    return out
sim_garch = simulate_garch(len(r_train))
print("GARCH pronto — VaR/CVaR e path simulato generati.")


## 6. Benchmark 2 — QuantGAN (TCN)

QuantGAN (Wiese et al., 2020) usa reti **TCN** (Temporal Convolutional Network, convoluzioni causali dilatate)
per generatore e discriminatore. Qui ne implementiamo una versione compatta che lavora sui rendimenti
standardizzati (la trasformazione di Lambert-W dell'articolo originale è lasciata come estensione).


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

class TCNBlock(nn.Module):
    def __init__(self, cin, cout, k=2, d=1):
        super().__init__()
        self.pad = (k - 1) * d                      # padding causale
        self.conv = nn.Conv1d(cin, cout, k, dilation=d)
        self.act  = nn.PReLU(cout)
        self.down = nn.Conv1d(cin, cout, 1) if cin != cout else None
    def forward(self, x):
        y = self.conv(nn.functional.pad(x, (self.pad, 0)))
        y = self.act(y)
        res = x if self.down is None else self.down(x)
        return y + res

class TCN(nn.Module):
    def __init__(self, cin, cout, ch=32, dilations=(1, 2, 4, 8, 16)):
        super().__init__()
        layers, c = [], cin
        for d in dilations:
            layers.append(TCNBlock(c, ch, k=2, d=d)); c = ch
        self.net  = nn.Sequential(*layers)
        self.head = nn.Conv1d(ch, cout, 1)
    def forward(self, x):
        return self.head(self.net(x))

NZ = 3                                              # canali di rumore in ingresso al generatore
G = TCN(NZ, 1).to(DEVICE)
D = TCN(1, 1).to(DEVICE)

Xr = torch.tensor(W[:, None, :], dtype=torch.float32)   # (N,1,L)
loader = DataLoader(TensorDataset(Xr), batch_size=128, shuffle=True, drop_last=True)

optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.9))
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.9))
bce  = nn.BCEWithLogitsLoss()

def gen_noise(b): return torch.randn(b, NZ, L, device=DEVICE)
print(f"QuantGAN — parametri G: {sum(p.numel() for p in G.parameters())}, "
      f"D: {sum(p.numel() for p in D.parameters())}")


In [ ]:
EPOCHS_GAN = 40        # aumenta a 150-300 per risultati definitivi
hist = []
for ep in range(EPOCHS_GAN):
    dl = gl = 0.0
    for (xb,) in loader:
        xb = xb.to(DEVICE); b = xb.size(0)
        # --- D ---
        optD.zero_grad()
        out_r = D(xb).mean(dim=2).squeeze(1)
        xf = G(gen_noise(b)).detach()
        out_f = D(xf).mean(dim=2).squeeze(1)
        lossD = bce(out_r, torch.ones_like(out_r) * 0.9) + bce(out_f, torch.zeros_like(out_f))
        lossD.backward(); optD.step()
        # --- G ---
        optG.zero_grad()
        xf = G(gen_noise(b))
        out = D(xf).mean(dim=2).squeeze(1)
        lossG = bce(out, torch.ones_like(out))
        lossG.backward(); optG.step()
        dl += lossD.item(); gl += lossG.item()
    hist.append((dl/len(loader), gl/len(loader)))
    if (ep+1) % 10 == 0 or ep == 0:
        print(f"epoca {ep+1:3d}/{EPOCHS_GAN}  lossD={hist[-1][0]:.3f}  lossG={hist[-1][1]:.3f}")

@torch.no_grad()
def gan_generate(n_windows=2000):
    G.eval()
    x = G(gen_noise(n_windows)).cpu().numpy()[:, 0, :]   # (n,L) standardizzate
    G.train(); return x

W_gan = gan_generate()
z_gan = W_gan.reshape(-1)                                # pool di innovazioni
q_gan, es_gan = tail_shape(z_gan)
VaR_gan, CVaR_gan = var_cvar_series(sigma_ewma_test, q_gan, es_gan, mu=mu)
sim_gan = (W_gan.reshape(-1) * sd + mu)[:len(r_train)]   # in unita' di rendimento, per i fatti stilizzati
print("QuantGAN pronto —", W_gan.shape, "finestre generate.")


In [ ]:
# --- Cella di TEST: output del generatore ben formato ---
assert W_gan.shape[1] == L
assert np.isfinite(W_gan).all(), "Output NaN/inf: ridurre lr o usare WGAN-GP"
print("OK — QuantGAN genera finestre finite della lunghezza attesa.")


## 7. Modello proposto — Diffusion (DDPM)

**Processo forward** (aggiunge rumore, nessun parametro):
$$x_k = \sqrt{\bar\alpha_k}\,x_0 + \sqrt{1-\bar\alpha_k}\,\epsilon,\qquad \epsilon\sim\mathcal N(0,I).$$
**Processo reverse** (si impara): una rete $\epsilon_\theta(x_k,k)$ predice il rumore; la loss è
$\;\mathbb E\lVert \epsilon - \epsilon_\theta(x_k,k)\rVert^2.$
Il campionamento parte da $x_T\sim\mathcal N(0,I)$ e denoisa fino a $x_0$.


In [ ]:
T = 200
betas = torch.linspace(1e-4, 0.02, T, device=DEVICE)
alphas = 1.0 - betas
abar = torch.cumprod(alphas, dim=0)

def timestep_emb(t, dim=32):
    half = dim // 2
    freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / half)
    a = t[:, None].float() * freqs[None]
    return torch.cat([torch.sin(a), torch.cos(a)], dim=1)

class EpsNet(nn.Module):
    def __init__(self, ch=64, temb=32):
        super().__init__()
        self.temb = temb
        self.tproj = nn.Sequential(nn.Linear(temb, ch), nn.SiLU(), nn.Linear(ch, ch))
        self.inp  = nn.Conv1d(1, ch, 3, padding=1)
        self.mid  = nn.ModuleList([nn.Conv1d(ch, ch, 3, padding=1, dilation=1),
                                   nn.Conv1d(ch, ch, 3, padding=2, dilation=2),
                                   nn.Conv1d(ch, ch, 3, padding=4, dilation=4)])
        self.act  = nn.SiLU()
        self.out  = nn.Conv1d(ch, 1, 3, padding=1)
    def forward(self, x, t):
        h = self.inp(x)
        temb = self.tproj(timestep_emb(t, self.temb))[:, :, None]
        for conv in self.mid:
            h = self.act(conv(h) + temb)
        return self.out(h)

eps_net = EpsNet().to(DEVICE)
optE = torch.optim.Adam(eps_net.parameters(), lr=2e-4)
print("Diffusion — parametri:", sum(p.numel() for p in eps_net.parameters()))


In [ ]:
EPOCHS_DIFF = 60       # aumenta a 200-400 per risultati definitivi
loss_hist = []
for ep in range(EPOCHS_DIFF):
    tot = 0.0
    for (xb,) in loader:
        xb = xb.to(DEVICE); b = xb.size(0)
        t = torch.randint(0, T, (b,), device=DEVICE)
        noise = torch.randn_like(xb)
        ab = abar[t][:, None, None]
        x_noisy = torch.sqrt(ab) * xb + torch.sqrt(1 - ab) * noise
        pred = eps_net(x_noisy, t)
        loss = ((noise - pred) ** 2).mean()
        optE.zero_grad(); loss.backward(); optE.step()
        tot += loss.item()
    loss_hist.append(tot / len(loader))
    if (ep+1) % 10 == 0 or ep == 0:
        print(f"epoca {ep+1:3d}/{EPOCHS_DIFF}  MSE={loss_hist[-1]:.4f}")

@torch.no_grad()
def diffusion_sample(n_windows=2000):
    eps_net.eval()
    x = torch.randn(n_windows, 1, L, device=DEVICE)
    for k in reversed(range(T)):
        tk = torch.full((n_windows,), k, device=DEVICE, dtype=torch.long)
        beta_k, ab_k, a_k = betas[k], abar[k], alphas[k]
        pred = eps_net(x, tk)
        mean = (x - beta_k / torch.sqrt(1 - ab_k) * pred) / torch.sqrt(a_k)
        x = mean + (torch.sqrt(beta_k) * torch.randn_like(x) if k > 0 else 0.0)
    eps_net.train()
    return x.cpu().numpy()[:, 0, :]

W_diff = diffusion_sample()
z_diff = W_diff.reshape(-1)
q_d, es_d = tail_shape(z_diff)
VaR_d, CVaR_d = var_cvar_series(sigma_ewma_test, q_d, es_d, mu=mu)
sim_diff = (W_diff.reshape(-1) * sd + mu)[:len(r_train)]
print("Diffusion pronto —", W_diff.shape, "finestre generate.")


In [ ]:
# --- Cella di TEST: output del diffusion ben formato ---
assert W_diff.shape[1] == L
assert np.isfinite(W_diff).all(), "Output NaN/inf: aumentare le epoche o ridurre lr"
print("OK — Diffusion genera finestre finite della lunghezza attesa.")


## 8. Risultati — piano 1: realismo (fatti stilizzati)

Confrontiamo le proprietà statistiche delle serie **generate** con quelle dei rendimenti **reali del train**.
Il modello migliore è quello le cui colonne sono più vicine alla colonna *reale*.


In [ ]:
tab_sf = pd.concat([
    stylized_facts(r_train.values, "reale (train)"),
    stylized_facts(sim_garch,      "GARCH-t"),
    stylized_facts(sim_gan,        "QuantGAN"),
    stylized_facts(sim_diff,       "Diffusion"),
], axis=1).T
tab_sf.round(4)


In [ ]:
# QQ-plot dei rendimenti generati contro i reali
def qq(ax, real, gen, name):
    qs = np.linspace(0.01, 0.99, 99)
    ax.plot(np.quantile(real, qs), np.quantile(gen, qs), ".", ms=4)
    lim = [min(real.min(), gen.min()), max(real.max(), gen.max())]
    ax.plot(lim, lim, "r--", lw=1); ax.set_title(name)
    ax.set_xlabel("quantili reali"); ax.set_ylabel("quantili generati")

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
qq(ax[0], r_train.values, sim_garch, "GARCH-t")
qq(ax[1], r_train.values, sim_gan,   "QuantGAN")
qq(ax[2], r_train.values, sim_diff,  "Diffusion")
plt.tight_layout(); plt.show()


In [ ]:
# ACF dei rendimenti al quadrato: volatility clustering
fig, ax = plt.subplots(figsize=(9, 4)); lags = 20
for x, nm in [(r_train.values, "reale"), (sim_garch, "GARCH-t"),
              (sim_gan, "QuantGAN"), (sim_diff, "Diffusion")]:
    ax.plot(range(1, lags+1), acf(x**2, nlags=lags, fft=True)[1:], marker="o", ms=3, label=nm)
ax.axhline(0, color="k", lw=0.5); ax.set_title("ACF dei rendimenti^2"); ax.legend()
ax.set_xlabel("lag"); plt.tight_layout(); plt.show()


## 9. Risultati — piano 2: VaR/CVaR e backtesting out-of-sample

Test sul periodo **2019→oggi**, mai visto in addestramento. Interpretazione dei p-value (soglia 5%):

- **Kupiec p > 0.05** → il numero di violazioni è coerente con il 99% (buono);
- **Christoffersen p > 0.05** → le violazioni sono indipendenti, non a grappoli (buono);
- **CC p > 0.05** → copertura condizionale corretta (il test complessivo).


In [ ]:
results = pd.concat([
    backtest(r_test.values, VaR_g,   CVaR_g,   name="GARCH-t"),
    backtest(r_test.values, VaR_gan, CVaR_gan, name="QuantGAN"),
    backtest(r_test.values, VaR_d,   CVaR_d,   name="Diffusion"),
], axis=1).T
results.round(4)


In [ ]:
# Rendimenti del test vs VaR 99% dei tre modelli, con violazioni evidenziate
fig, ax = plt.subplots(figsize=(12, 5))
idx = r_test.index
ax.plot(idx, r_test.values, color="0.6", lw=0.5, label="rendimenti test")
for VaR, nm, c in [(VaR_g, "GARCH-t", "C0"), (VaR_gan, "QuantGAN", "C1"), (VaR_d, "Diffusion", "C2")]:
    ax.plot(idx, VaR, lw=1.1, color=c, label=f"VaR99 {nm}")
    br = r_test.values < VaR
    ax.scatter(idx[br], r_test.values[br], s=14, color=c, zorder=5)
ax.set_title("VaR 99% a 1 giorno e violazioni (out-of-sample)"); ax.legend(ncol=2)
plt.tight_layout(); plt.show()


In [ ]:
# --- Cella di TEST: forme coerenti e VaR nel verso giusto (negativo) ---
for nm, VaR in [("GARCH-t", VaR_g), ("QuantGAN", VaR_gan), ("Diffusion", VaR_d)]:
    assert len(VaR) == len(r_test), f"{nm}: lunghezza VaR != test"
    assert np.median(VaR) < 0, f"{nm}: il VaR di coda sinistra deve essere negativo"
print("OK — tutte le serie VaR sono allineate al test e negative come atteso.")


## 10. Conclusioni e come leggerle

Confronto atteso, da commentare con i **numeri effettivi** prodotti dalle celle sopra:

- **GARCH-t** — grazie alla volatilità condizionale reagisce in fretta agli shock: di solito supera bene il test di **indipendenza** di Christoffersen (poche violazioni a grappolo). È il riferimento da battere sul VaR.
- **QuantGAN** — cattura bene la *forma* delle code (curtosi, QQ-plot) ma l'addestramento avversariale è instabile; con poche epoche può sotto/sovra-stimare la coda. Con più epoche e la trasformazione di Lambert-W migliora.
- **Diffusion** — training stabile (loss MSE monotòna) e, se le epoche sono sufficienti, code realistiche; combinato con la scala EWMA tende a produrre un VaR competitivo e ben calibrato (Kupiec + CC).

**Messaggio della tesi.** Un diffusion model impara la distribuzione dei rendimenti da dati reali, ne riproduce i fatti stilizzati e, agganciato a una scala di volatilità condizionale, fornisce stime di VaR/CVaR che reggono il backtesting out-of-sample al pari o meglio dei benchmark GARCH-t e QuantGAN.

**Estensioni naturali:** (1) generazione *condizionale* end-to-end (il diffusion produce direttamente il rendimento del giorno successivo dato lo storico, senza EWMA); (2) multi-asset con matrice di correlazione; (3) mercati energetici (`CL=F`, `NG=F`); (4) QuantGAN con Lambert-W e WGAN-GP; (5) più epoche e tuning degli iperparametri.

> **Nota metodologica.** Con default a poche epoche i risultati sono *illustrativi* e servono a verificare che l'intera pipeline giri. Per la tesi: GPU, `EPOCHS_GAN ≈ 200`, `EPOCHS_DIFF ≈ 300`, ed eventualmente più ripetizioni con seed diversi per stimare la variabilità.
